In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

sys.path.append("../../utils")

from utils import (
    cargar_dataset,
    guardar_dataset_csv,
    resumen_clases
)

In [ ]:
# =========================
# CONFIGURACIÓN GENERAL
# =========================

DATASET_NAME = "CIC17"

LABEL_COL = "LABEL"

# Dataset de entrada
INPUT_DATASET_VERSION = "CIC17__split__v1"
INPUT_DIR = f"../../../02_datasets/processed/{INPUT_DATASET_VERSION}"

TRAIN_FILENAME = f"{INPUT_DATASET_VERSION}__train.csv"
TEST_FILENAME = f"{INPUT_DATASET_VERSION}__test.csv"

# Dataset de salida
OUTPUT_DATASET_VERSION = "CIC17__seleccion__v1"
OUTPUT_DIR = f"../../../02_datasets/processed/{OUTPUT_DATASET_VERSION}"

TRAIN_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__train.csv"
TEST_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__test.csv"

LOADINGS_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__pca_loadings.csv"
REPORT_OUTPUT_FILENAME = f"{OUTPUT_DATASET_VERSION}__report.json"

# PCA
N_COMPONENTS_ANALISIS = 5
TOP_N_POR_COMPONENTE = 10

RANDOM_STATE = 42

In [ ]:
df_train = cargar_dataset(
    nombre_dataset=TRAIN_FILENAME,
    ruta_base=INPUT_DIR
)

df_test = cargar_dataset(
    nombre_dataset=TEST_FILENAME,
    ruta_base=INPUT_DIR
)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

display(df_train.head())

In [ ]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No existe la columna {LABEL_COL} en train.")

if LABEL_COL not in df_test.columns:
    raise ValueError(f"No existe la columna {LABEL_COL} en test.")

print("Columna LABEL encontrada correctamente.")
print("Columnas train:", df_train.shape[1])
print("Columnas test:", df_test.shape[1])

In [ ]:
print("Distribución de clases en train:")
display(resumen_clases(df_train, label_col=LABEL_COL))

print("Distribución de clases en test:")
display(resumen_clases(df_test, label_col=LABEL_COL))

In [ ]:
X_train = df_train.drop(columns=[LABEL_COL])
y_train = df_train[LABEL_COL].copy()

X_test = df_test.drop(columns=[LABEL_COL])
y_test = df_test[LABEL_COL].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

non_numeric_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

print(f"Columnas numéricas usadas en PCA: {len(numeric_cols)}")
print(f"Columnas no numéricas excluidas: {len(non_numeric_cols)}")

if len(non_numeric_cols) > 0:
    print("Columnas no numéricas:")
    print(non_numeric_cols)

In [ ]:
missing_in_test = [
    col for col in numeric_cols
    if col not in X_test.columns
]

if len(missing_in_test) > 0:
    raise ValueError(f"Columnas de train que no están en test: {missing_in_test}")

X_train_num = X_train[numeric_cols].copy()
X_test_num = X_test[numeric_cols].copy()

print("X_train_num:", X_train_num.shape)
print("X_test_num:", X_test_num.shape)

In [ ]:
scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

print("RobustScaler aplicado correctamente.")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

In [ ]:
pca = PCA(
    n_components=N_COMPONENTS_ANALISIS,
    random_state=RANDOM_STATE
)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("PCA calculado correctamente.")
print("X_train_pca:", X_train_pca.shape)
print("X_test_pca:", X_test_pca.shape)

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Varianza explicada por componente:")

for i, var in enumerate(explained_variance, start=1):
    print(f"PC{i}: {var:.6f} ({var * 100:.2f}%)")

print()
print(f"Varianza acumulada con las {N_COMPONENTS_ANALISIS} primeras componentes:")
print(f"{cumulative_variance[-1]:.6f} ({cumulative_variance[-1] * 100:.2f}%)")

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=numeric_cols,
    columns=[f"PC{i}" for i in range(1, N_COMPONENTS_ANALISIS + 1)]
)

display(loadings.head())

In [ ]:
features_por_componente = {}

for pc in loadings.columns:
    top_features = (
        loadings[pc]
        .abs()
        .sort_values(ascending=False)
        .head(TOP_N_POR_COMPONENTE)
    )
    
    features_por_componente[pc] = top_features.index.tolist()
    
    print(f"\n===== {pc} =====")
    display(top_features.to_frame(name="peso_absoluto"))

In [ ]:
selected_features = []

for pc, features in features_por_componente.items():
    for feature in features:
        if feature not in selected_features:
            selected_features.append(feature)

print(f"Número total de columnas seleccionadas: {len(selected_features)}")
print()

for feature in selected_features:
    print("-", feature)

In [ ]:
df_train_reduced = df_train[selected_features + [LABEL_COL]].copy()
df_test_reduced = df_test[selected_features + [LABEL_COL]].copy()

print("Train original:", df_train.shape)
print("Train reducido:", df_train_reduced.shape)

print("Test original:", df_test.shape)
print("Test reducido:", df_test_reduced.shape)

display(df_train_reduced.head())

In [ ]:
print("Distribución de clases en train reducido:")
display(resumen_clases(df_train_reduced, label_col=LABEL_COL))

print("Distribución de clases en test reducido:")
display(resumen_clases(df_test_reduced, label_col=LABEL_COL))

In [ ]:
guardar_dataset_csv(
    df=df_train_reduced,
    nombre_archivo=TRAIN_OUTPUT_FILENAME,
    ruta=OUTPUT_DIR
)

guardar_dataset_csv(
    df=df_test_reduced,
    nombre_archivo=TEST_OUTPUT_FILENAME,
    ruta=OUTPUT_DIR
)

print("Datasets reducidos guardados correctamente:")
print(os.path.join(OUTPUT_DIR, TRAIN_OUTPUT_FILENAME))
print(os.path.join(OUTPUT_DIR, TEST_OUTPUT_FILENAME))

In [ ]:
loadings_to_save = loadings.reset_index().rename(columns={"index": "FEATURE"})

guardar_dataset_csv(
    df=loadings_to_save,
    nombre_archivo=LOADINGS_OUTPUT_FILENAME,
    ruta=OUTPUT_DIR
)

print("Loadings guardados correctamente:")
print(os.path.join(OUTPUT_DIR, LOADINGS_OUTPUT_FILENAME))

In [ ]:
report = {
    "dataset": DATASET_NAME,
    "input_dataset_version": INPUT_DATASET_VERSION,
    "output_dataset_version": OUTPUT_DATASET_VERSION,
    "input_dir": INPUT_DIR,
    "output_dir": OUTPUT_DIR,
    "train_filename": TRAIN_FILENAME,
    "test_filename": TEST_FILENAME,
    "train_output_filename": TRAIN_OUTPUT_FILENAME,
    "test_output_filename": TEST_OUTPUT_FILENAME,
    "label_col": LABEL_COL,
    "n_components_analisis": N_COMPONENTS_ANALISIS,
    "top_n_por_componente": TOP_N_POR_COMPONENTE,
    "random_state": RANDOM_STATE,
    "train_shape_original": list(df_train.shape),
    "test_shape_original": list(df_test.shape),
    "train_shape_reduced": list(df_train_reduced.shape),
    "test_shape_reduced": list(df_test_reduced.shape),
    "num_numeric_cols_used_for_pca": len(numeric_cols),
    "num_non_numeric_cols_excluded": len(non_numeric_cols),
    "non_numeric_cols_excluded": non_numeric_cols,
    "num_selected_features": len(selected_features),
    "selected_features": selected_features,
    "features_por_componente": features_por_componente,
    "explained_variance_ratio": explained_variance.tolist(),
    "cumulative_variance_ratio": cumulative_variance.tolist()
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

report_path = os.path.join(OUTPUT_DIR, REPORT_OUTPUT_FILENAME)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

print("Reporte guardado correctamente:")
print(report_path)

In [ ]:
print("======================================")
print("PREPROCESAMIENTO FINALIZADO")
print("======================================")
print()
print("Dataset de entrada:")
print(INPUT_DATASET_VERSION)
print()
print("Dataset de salida:")
print(OUTPUT_DATASET_VERSION)
print()
print("Train original:", df_train.shape)
print("Train reducido:", df_train_reduced.shape)
print()
print("Test original:", df_test.shape)
print("Test reducido:", df_test_reduced.shape)
print()
print(f"Componentes PCA analizadas: {N_COMPONENTS_ANALISIS}")
print(f"Top variables por componente: {TOP_N_POR_COMPONENTE}")
print(f"Variables finales seleccionadas: {len(selected_features)}")
print()
print("Archivos generados:")
print("-", os.path.join(OUTPUT_DIR, TRAIN_OUTPUT_FILENAME))
print("-", os.path.join(OUTPUT_DIR, TEST_OUTPUT_FILENAME))
print("-", os.path.join(OUTPUT_DIR, LOADINGS_OUTPUT_FILENAME))
print("-", report_path)